# Thesis Figures

Generates all static thesis charts and plates. Run each section independently — sections do not depend on each other.

Outputs land in `results/figures/graphs/` and `results/figures/plates/` and are committed to the repo.

| Section | What it shows | Output file |
|---|---|---|
| Behavior F1 Comparison | Macro-F1 across all 5 VideoMAE configs (v2) | `graphs/behavior_f1_comparison.{pdf,png}` |
| Cross-Domain F1 Comparison | In-domain vs cross-domain penalty per source domain | `graphs/cross_domain_f1_comparison.{pdf,png}` |
| Activity Budget + Timeline | Stacked budget bar + Gantt timeline for a CVB clip | `graphs/activity_budget_timeline.{pdf,png}` |
| Perturbation Illustrations | 5 perturbation types × clean/low/high severity | `plates/perturbation_illustrations.{pdf,png}` |
| Tracking Identity Strip | 4 consecutive CVB frames showing OC-SORT track continuity | `plates/tracking_identity_strip.{pdf,png}` |
| RF-DETR-Seg Training Comparison | Config A vs B: loss, mAP, P/R/F1 (9 charts) | `graphs/rfdetr_seg_training/*.png` |

**Run from repo root** (or ensure `REPO_ROOT` resolves correctly via git):
```bash
source ~/miniconda3/etc/profile.d/conda.sh && conda activate cattletransformer
jupyter notebook notebooks/thesis_figures.ipynb
```

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

%matplotlib inline

REPO_ROOT = Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"]).decode().strip()
)
sys.path.insert(0, str(REPO_ROOT / "src"))

GRAPHS_DIR = REPO_ROOT / "results" / "figures" / "graphs"
PLATES_DIR = REPO_ROOT / "results" / "figures" / "plates"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)
PLATES_DIR.mkdir(parents=True, exist_ok=True)

SHARED_STYLE = {
    "font.family":       "serif",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "axes.spines.top":   False,
    "axes.spines.right": False,
}

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"Outputs → {GRAPHS_DIR} and {PLATES_DIR}")

---
## Behavior F1 Comparison

Macro-F1 bar chart across all five VideoMAE training configurations (v2 results).
Data is hardcoded from `results/behavior/v1_v2_comparison.csv` — frozen thesis numbers.

In [ ]:
plt.rcParams.update(SHARED_STYLE)

CONFIGS = [
    ("Config 1\n(CBVD-5\nin-domain)",  0.451127, "in-domain"),
    ("Config 2\n(CVB\nin-domain)",      0.776986, "in-domain"),
    ("Config 3\n(CBVD-5→CVB)",          0.172232, "cross-domain"),
    ("Config 4\n(CVB→CBVD-5)",          0.225310, "cross-domain"),
    ("Config 5\n(Combined)",             0.750660, "combined"),
]
COMBINED_BASELINE = 0.750660

COLORS      = {"in-domain": "#185FA5", "cross-domain": "#C0392B", "combined": "#1D9E75"}
EDGE_COLORS = {"in-domain": "#0C447C", "cross-domain": "#7B241C", "combined": "#0F6E56"}

fig, ax = plt.subplots(figsize=(7.5, 4.2))

labels = [c[0] for c in CONFIGS]
values = [c[1] for c in CONFIGS]
groups = [c[2] for c in CONFIGS]
x      = np.arange(len(CONFIGS))

bars = ax.bar(
    x, values, width=0.52,
    color=[COLORS[g] for g in groups],
    edgecolor=[EDGE_COLORS[g] for g in groups],
    linewidth=1.2, zorder=3,
)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, val + 0.012, f"{val:.3f}",
        ha="center", va="bottom", fontsize=8.5, fontfamily="serif", color="#222",
    )

ax.axhline(COMBINED_BASELINE, color="#555", linewidth=1.1, linestyle="--", zorder=2)
ax.text(
    2.5, COMBINED_BASELINE + 0.018,
    f"Combined baseline ({COMBINED_BASELINE:.3f})",
    ha="center", va="bottom", fontsize=8, fontfamily="serif", color="#555", style="italic",
)

ax.set_xticks(x)
ax.set_xticklabels(labels, multialignment="center")
ax.set_ylabel("Macro-F1", labelpad=6)
ax.set_ylim(0, 1.05)
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.yaxis.grid(True, linestyle=":", linewidth=0.6, color="#ccc", zorder=0)
ax.set_axisbelow(True)

ax.legend(
    handles=[
        mpatches.Patch(facecolor=COLORS[g], edgecolor=EDGE_COLORS[g], label=g.capitalize())
        for g in ["in-domain", "cross-domain", "combined"]
    ],
    loc="upper right", fontsize=8.5, framealpha=0.85, edgecolor="#ccc",
    prop={"family": "serif"},
)

fig.tight_layout(pad=0.6)
fig.savefig(GRAPHS_DIR / "behavior_f1_comparison.pdf", dpi=300)
fig.savefig(GRAPHS_DIR / "behavior_f1_comparison.png", dpi=300)
plt.show()
print(f"Saved → {GRAPHS_DIR}/behavior_f1_comparison.{{pdf,png}}")

---
## Cross-Domain F1 Comparison

Grouped bar chart: each pair shows in-domain baseline vs cross-domain transfer for the same
source domain, with macro-F1 penalty annotated between bars.
Data hardcoded from `results/behavior/v1_v2_comparison.csv`.

In [ ]:
plt.rcParams.update(SHARED_STYLE)

# Group 1: CBVD-5 source — in-domain (Config 1) vs transfer to CVB (Config 3)
# Group 2: CVB source   — in-domain (Config 2) vs transfer to CBVD-5 (Config 4)
IN_DOMAIN    = [0.451127, 0.776986]
CROSS_DOMAIN = [0.172232, 0.225310]
GROUP_LABELS = [
    "CBVD-5 domain\n(CBVD-5 → CBVD-5 vs. CBVD-5 → CVB)",
    "CVB domain\n(CVB → CVB vs. CVB → CBVD-5)",
]

COLOR_IN    = "#185FA5";  EDGE_IN    = "#0C447C"
COLOR_CROSS = "#C0392B";  EDGE_CROSS = "#7B241C"

fig, ax = plt.subplots(figsize=(7.0, 4.2))

x      = np.arange(len(GROUP_LABELS))
bar_w  = 0.32
offset = bar_w / 2 + 0.03

bars_in    = ax.bar(x - offset, IN_DOMAIN,    width=bar_w, color=COLOR_IN,    edgecolor=EDGE_IN,
                    linewidth=1.2, zorder=3, label="In-domain baseline")
bars_cross = ax.bar(x + offset, CROSS_DOMAIN, width=bar_w, color=COLOR_CROSS, edgecolor=EDGE_CROSS,
                    linewidth=1.2, zorder=3, label="Cross-domain transfer", hatch="//", alpha=0.85)

for bar, val in zip(bars_in, IN_DOMAIN):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.013, f"{val:.3f}",
            ha="center", va="bottom", fontsize=8.5, fontfamily="serif", color="#222")
for bar, val in zip(bars_cross, CROSS_DOMAIN):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.013, f"{val:.3f}",
            ha="center", va="bottom", fontsize=8.5, fontfamily="serif", color="#222")

for i, (v_in, v_cross) in enumerate(zip(IN_DOMAIN, CROSS_DOMAIN)):
    penalty_pp = (v_in - v_cross) * 100
    mid_y = (v_in + v_cross) / 2
    ax.annotate(
        f"−{penalty_pp:.0f} pp", xy=(x[i], mid_y), xytext=(x[i], mid_y),
        ha="center", va="center", fontsize=8.5, fontfamily="serif",
        color=COLOR_CROSS, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                  edgecolor=COLOR_CROSS, linewidth=0.8, alpha=0.9),
    )
    ax.annotate("", xy=(x[i] - offset + bar_w / 2, v_in), xytext=(x[i], mid_y + 0.025),
                arrowprops=dict(arrowstyle="-", color=COLOR_CROSS, lw=0.8, linestyle="dashed"))
    ax.annotate("", xy=(x[i] + offset - bar_w / 2, v_cross), xytext=(x[i], mid_y - 0.025),
                arrowprops=dict(arrowstyle="-", color=COLOR_CROSS, lw=0.8, linestyle="dashed"))

ax.set_xticks(x)
ax.set_xticklabels(GROUP_LABELS, multialignment="center")
ax.set_ylabel("Macro-F1", labelpad=6)
ax.set_ylim(0, 1.02)
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.yaxis.grid(True, linestyle=":", linewidth=0.6, color="#ccc", zorder=0)
ax.set_axisbelow(True)
ax.legend(
    handles=[
        mpatches.Patch(facecolor=COLOR_IN,    edgecolor=EDGE_IN,    label="In-domain baseline"),
        mpatches.Patch(facecolor=COLOR_CROSS, edgecolor=EDGE_CROSS, hatch="//", alpha=0.85,
                       label="Cross-domain transfer"),
    ],
    loc="upper right", fontsize=8.5, framealpha=0.9, edgecolor="#ccc", prop={"family": "serif"},
)

fig.tight_layout(pad=0.6)
fig.savefig(GRAPHS_DIR / "cross_domain_f1_comparison.pdf", dpi=300)
fig.savefig(GRAPHS_DIR / "cross_domain_f1_comparison.png", dpi=300)
plt.show()
print(f"Saved → {GRAPHS_DIR}/cross_domain_f1_comparison.{{pdf,png}}")

for label, v_in, v_cross in zip(GROUP_LABELS, IN_DOMAIN, CROSS_DOMAIN):
    print(f"  {label.split(chr(10))[0]}: in={v_in:.3f}, cross={v_cross:.3f}, penalty=−{(v_in-v_cross)*100:.0f} pp")

---
## Activity Budget + Timeline

Two-panel figure for CVB clip `0258_arm01_gopro1_20200323_034430_beh4_ani6_ins1_cut_3`:
- **Top:** horizontal stacked bar showing overall activity budget for the clip
- **Bottom:** Gantt timeline — one row per tracked animal, colour = behavior class

Reads from `results/behavior_analytics/timelines/cvb/<video_id>/track_*.csv`.

In [ ]:
import pandas as pd
from matplotlib.gridspec import GridSpec

plt.rcParams.update({**SHARED_STYLE, "font.size": 9, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8})

VIDEO_ID     = "0258_arm01_gopro1_20200323_034430_beh4_ani6_ins1_cut_3"
TIMELINE_DIR = REPO_ROOT / "results" / "analytics" / "timelines" / "cvb" / VIDEO_ID

BEHAVIOR_ORDER  = ["Standing", "Lying", "Foraging", "Drinking", "Ruminating", "Grooming", "Other"]
BEHAVIOR_COLORS = {
    "Standing":  "#7F8C8D",
    "Lying":     "#2980B9",
    "Foraging":  "#27AE60",
    "Drinking":  "#16A085",
    "Ruminating":"#E67E22",
    "Grooming":  "#8E44AD",
    "Other":     "#BDC3C7",
}

track_files = sorted(TIMELINE_DIR.glob("track_*.csv"))
if not track_files:
    raise FileNotFoundError(f"No timeline CSVs in {TIMELINE_DIR}")
track_dfs = {tf.stem: pd.read_csv(tf) for tf in track_files if not pd.read_csv(tf).empty}
tracks    = sorted(track_dfs.keys())
n_tracks  = len(tracks)
clip_end  = max(df["end_sec"].max() for df in track_dfs.values())

all_segs = pd.concat(track_dfs.values(), ignore_index=True)
total_sec = all_segs["duration_sec"].sum()
budget    = (all_segs.groupby("label_name")["duration_sec"].sum() / total_sec * 100).to_dict()

fig = plt.figure(figsize=(10.0, 5.2))
gs  = GridSpec(2, 1, figure=fig, height_ratios=[1, 3], hspace=0.38)
ax_budget   = fig.add_subplot(gs[0])
ax_timeline = fig.add_subplot(gs[1])

# Activity budget bar
left = 0.0
for beh in BEHAVIOR_ORDER:
    pct = budget.get(beh, 0.0)
    if pct == 0:
        continue
    ax_budget.barh(0, pct, left=left, height=0.55,
                   color=BEHAVIOR_COLORS[beh], edgecolor="white", linewidth=0.6)
    if pct >= 4:
        ax_budget.text(left + pct / 2, 0, f"{pct:.0f}%",
                       ha="center", va="center", fontsize=7.5,
                       fontfamily="serif", color="white", fontweight="bold")
    left += pct
ax_budget.set_xlim(0, 100)
ax_budget.set_ylim(-0.6, 0.6)
ax_budget.set_xlabel("Time (%)", labelpad=4)
ax_budget.set_yticks([])
ax_budget.set_title("Activity budget (clip aggregate)", fontsize=9, fontfamily="serif", pad=4)
for spine in ["top", "right", "left"]:
    ax_budget.spines[spine].set_visible(False)

# Gantt timeline
ytick_labels = []
for row_idx, tid in enumerate(tracks):
    df    = track_dfs[tid]
    y     = n_tracks - 1 - row_idx
    bar_h = 0.65
    for _, seg in df.iterrows():
        ax_timeline.broken_barh(
            [(seg["start_sec"], seg["duration_sec"])], (y - bar_h / 2, bar_h),
            facecolors=BEHAVIOR_COLORS.get(seg["label_name"], "#999"),
            edgecolors="white", linewidth=0.4,
        )
    num = tid.replace("track_0", "").replace("track_", "")
    ytick_labels.append(f"Animal {int(num):02d}")
ax_timeline.set_yticks(range(n_tracks))
ax_timeline.set_yticklabels(list(reversed(ytick_labels)))
ax_timeline.set_xlim(0, clip_end)
ax_timeline.set_ylim(-0.6, n_tracks - 0.4)
ax_timeline.set_xlabel("Time (s)", labelpad=4)
ax_timeline.set_title("Behavior timeline", fontsize=9, fontfamily="serif", pad=4)
ax_timeline.xaxis.grid(True, linestyle=":", linewidth=0.5, color="#ddd", zorder=0)
ax_timeline.set_axisbelow(True)
for spine in ["top", "right"]:
    ax_timeline.spines[spine].set_visible(False)

present = [b for b in BEHAVIOR_ORDER if budget.get(b, 0) > 0]
fig.legend(
    handles=[mpatches.Patch(facecolor=BEHAVIOR_COLORS[b], edgecolor="#888",
                            linewidth=0.5, label=b) for b in present],
    loc="lower center", ncol=len(present), fontsize=8, framealpha=0.9,
    edgecolor="#ccc", bbox_to_anchor=(0.5, -0.04), prop={"family": "serif"},
)

fig.savefig(GRAPHS_DIR / "activity_budget_timeline.pdf", dpi=300, bbox_inches="tight")
fig.savefig(GRAPHS_DIR / "activity_budget_timeline.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved → {GRAPHS_DIR}/activity_budget_timeline.{{pdf,png}}")
print(f"\nClip: {VIDEO_ID}  |  {n_tracks} tracks  |  {clip_end:.1f} s")
for b in BEHAVIOR_ORDER:
    pct = budget.get(b, 0)
    if pct > 0:
        print(f"  {b:12s}: {pct:.1f}%")

---
## RF-DETR-Seg Training Comparison

Nine charts comparing **Config A** (lr=1e-4) vs **Config B** (lr=5e-5) RF-DETR-Seg-Medium
training runs from Phase 3b on HiPE1. Config B is the winning run — its checkpoint
`runs/seg_medium_lr5e5/checkpoint_best_ema.pth` underpins all v2 thesis results.

**Log files (JSON Lines, one object per training step):**
- `logs/train_rfdetr_seg_hipe1.log` — Config B (committed, the winning run)
- Config A log not committed (losing run); obtain from HiPE1 `_archive/` if needed.
  Set `LOG_A = None` below to skip Config A curves.

Outputs saved to `results/figures/graphs/rfdetr_seg_training/`.

In [ ]:
import json as _json
import warnings
import os
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings("ignore")

# ── Log paths ───────────────────────────────────────────────────────────────
LOG_B = REPO_ROOT / "logs" / "train_rfdetr_seg_hipe1.log"   # Config B — committed
LOG_A = None  # set to REPO_ROOT / "logs" / "train_rfdetr_seg_config_a.log" if available

RFDETR_DIR = GRAPHS_DIR / "rfdetr_seg_training"
RFDETR_DIR.mkdir(parents=True, exist_ok=True)

# ── Color palette ───────────────────────────────────────────────────────────
C_A = "#2563EB";   C_A_LIGHT = "#93C5FD";  C_EMA_A = "#1D4ED8"
C_B = "#DC2626";   C_B_LIGHT = "#FCA5A5";  C_EMA_B = "#991B1B"
BG = "#F8FAFC";    GRID_COLOR = "#E2E8F0"; TEXT_COLOR = "#1E293B"; ACCENT = "#F59E0B"

RFDETR_STYLE = {
    "font.family": "DejaVu Sans",
    "axes.facecolor": BG, "figure.facecolor": "white",
    "axes.edgecolor": "#CBD5E1", "axes.labelcolor": TEXT_COLOR,
    "axes.titlecolor": TEXT_COLOR, "xtick.color": TEXT_COLOR, "ytick.color": TEXT_COLOR,
    "grid.color": GRID_COLOR, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.labelsize": 11,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.fontsize": 9, "legend.framealpha": 0.9, "legend.edgecolor": "#CBD5E1",
}
plt.rcParams.update(RFDETR_STYLE)

# ── Helpers ─────────────────────────────────────────────────────────────────
def load_log(path):
    records = []
    if path is None or not path.exists():
        return records
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(_json.loads(line))
                except _json.JSONDecodeError:
                    pass
    return records

def extract(records, key, default=None):
    return [r.get(key, default) for r in records]

def extract_map(records, source="ema_test_results_json"):
    map50, map5095 = [], []
    for r in records:
        obj = r.get(source, {})
        cm = obj.get("class_map", []) if obj else []
        cattle = next((c for c in cm if c["class"] == "cattle"), None)
        map50.append(cattle["map@50"] * 100 if cattle else None)
        map5095.append(cattle["map@50:95"] * 100 if cattle else None)
    return map50, map5095

def extract_prf(records, source="ema_test_results_json"):
    precision, recall, f1 = [], [], []
    for r in records:
        obj = r.get(source, {})
        cm = obj.get("class_map", []) if obj else []
        cattle = next((c for c in cm if c["class"] == "cattle"), None)
        precision.append(cattle["precision"] * 100 if cattle else None)
        recall.append(cattle["recall"] * 100 if cattle else None)
        f1.append(cattle["f1_score"] * 100 if cattle else None)
    return precision, recall, f1

def style_ax(ax, title, xlabel="Epoch", ylabel=None, legend=True):
    ax.set_title(title, fontweight="bold", pad=10)
    ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(axis="y", linestyle="--", alpha=0.6)
    ax.grid(axis="x", linestyle=":", alpha=0.3)
    if legend:
        ax.legend(loc="best")

def annotate_last(ax, x, y, color, fmt="{:.2f}"):
    if y and y[-1] is not None:
        ax.annotate(fmt.format(y[-1]), xy=(x[-1], y[-1]), xytext=(4, 0),
                    textcoords="offset points", fontsize=8, color=color,
                    fontweight="bold", va="center")

def add_best_marker(ax, epochs, values, color):
    valid = [(e, v) for e, v in zip(epochs, values) if v is not None]
    if not valid:
        return
    best_e, best_v = max(valid, key=lambda x: x[1])
    ax.scatter([best_e], [best_v], s=120, color=color, zorder=5, marker="*",
               edgecolors="white", linewidths=0.8)
    ax.annotate(f" ★ {best_v:.2f}%", xy=(best_e, best_v), fontsize=8,
                color=color, fontweight="bold", xytext=(5, 3),
                textcoords="offset points")

# ── Data loading ────────────────────────────────────────────────────────────
print("Loading logs...")
recs_B = load_log(LOG_B)
recs_A = load_log(LOG_A)

epochs_B = [r["epoch"] + 1 for r in recs_B]
epochs_A = [r["epoch"] + 1 for r in recs_A]

train_loss_A = extract(recs_A, "train_loss");  test_loss_A = extract(recs_A, "test_loss")
ema_loss_A   = extract(recs_A, "ema_test_loss")
train_loss_B = extract(recs_B, "train_loss");  test_loss_B = extract(recs_B, "test_loss")
ema_loss_B   = extract(recs_B, "ema_test_loss")

det_map50_A,    det_map5095_A    = extract_map(recs_A, "test_results_json")
det_map50_A_ema, det_map5095_A_ema = extract_map(recs_A, "ema_test_results_json")
det_map50_B,    det_map5095_B    = extract_map(recs_B, "test_results_json")
det_map50_B_ema, det_map5095_B_ema = extract_map(recs_B, "ema_test_results_json")

seg_map50_A,    seg_map5095_A    = extract_map(recs_A, "test_results_json_masks")
seg_map50_A_ema, seg_map5095_A_ema = extract_map(recs_A, "ema_test_results_json_masks")
seg_map50_B,    seg_map5095_B    = extract_map(recs_B, "test_results_json_masks")
seg_map50_B_ema, seg_map5095_B_ema = extract_map(recs_B, "ema_test_results_json_masks")

prec_A, rec_A, f1_A = extract_prf(recs_A, "ema_test_results_json")
prec_B, rec_B, f1_B = extract_prf(recs_B, "ema_test_results_json")
seg_prec_A, seg_rec_A, seg_f1_A = extract_prf(recs_A, "ema_test_results_json_masks")
seg_prec_B, seg_rec_B, seg_f1_B = extract_prf(recs_B, "ema_test_results_json_masks")

def parse_time_minutes(t):
    if t is None: return None
    h, m, s = t.split(":"); return int(h) * 60 + int(m) + float(s) / 60
train_min_A = [parse_time_minutes(t) for t in extract(recs_A, "train_epoch_time")]
train_min_B = [parse_time_minutes(t) for t in extract(recs_B, "train_epoch_time")]

print(f"  Config A: {len(recs_A)} epochs {'(not loaded — LOG_A is None)' if not recs_A else ''}")
print(f"  Config B: {len(recs_B)} epochs")

In [ ]:
# ── Chart 1: Total Loss Overview ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training & Validation Loss — Config A vs Config B",
             fontsize=15, fontweight="bold", y=1.02)

ax = axes[0]
if recs_A:
    ax.plot(epochs_A, train_loss_A, color=C_A, lw=2, marker="o", ms=5, label="A Train")
    ax.plot(epochs_A, test_loss_A,  color=C_A, lw=2, marker="s", ms=5, ls="--", label="A Val")
    ax.fill_between(epochs_A, train_loss_A, test_loss_A, alpha=0.06, color=C_A)
    annotate_last(ax, epochs_A, train_loss_A, C_A)
ax.plot(epochs_B, train_loss_B, color=C_B, lw=2, marker="o", ms=5, label="B Train")
ax.plot(epochs_B, test_loss_B,  color=C_B, lw=2, marker="s", ms=5, ls="--", label="B Val")
ax.fill_between(epochs_B, train_loss_B, test_loss_B, alpha=0.06, color=C_B)
annotate_last(ax, epochs_B, train_loss_B, C_B)
style_ax(ax, "Total Loss per Epoch", ylabel="Loss")

ax = axes[1]
if recs_A:
    ax.plot(epochs_A, ema_loss_A, color=C_EMA_A, lw=2.5, marker="D", ms=5, label="A — EMA Val")
    ax.fill_between(epochs_A, ema_loss_A, alpha=0.12, color=C_A)
    annotate_last(ax, epochs_A, ema_loss_A, C_EMA_A)
ax.plot(epochs_B, ema_loss_B, color=C_EMA_B, lw=2.5, marker="D", ms=5, label="B — EMA Val")
ax.fill_between(epochs_B, ema_loss_B, alpha=0.12, color=C_B)
annotate_last(ax, epochs_B, ema_loss_B, C_EMA_B)
style_ax(ax, "EMA Validation Loss Comparison", ylabel="EMA Loss")

plt.tight_layout()
fig.savefig(RFDETR_DIR / "01_total_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RFDETR_DIR}/01_total_loss.png")

# ── Chart 2: Component Loss Breakdown ───────────────────────────────────────
COMPONENTS = {
    "CE Loss":        ("train_loss_ce",      "test_loss_ce"),
    "BBox Loss":      ("train_loss_bbox",    "test_loss_bbox"),
    "GIoU Loss":      ("train_loss_giou",    "test_loss_giou"),
    "Mask CE Loss":   ("train_loss_mask_ce", "test_loss_mask_ce"),
    "Mask Dice Loss": ("train_loss_mask_dice","test_loss_mask_dice"),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Loss Component Breakdown — Train vs Validation", fontsize=15, fontweight="bold")

import matplotlib.patches as _mp
for idx, (name, (tk, vk)) in enumerate(COMPONENTS.items()):
    row, col = divmod(idx, 3)
    ax = axes[row][col]
    if recs_A:
        ax.plot(epochs_A, extract(recs_A, tk), color=C_A, lw=2, marker="o", ms=4, label="A Train")
        ax.plot(epochs_A, extract(recs_A, vk), color=C_A, lw=1.5, marker="s", ms=4, ls="--",
                alpha=0.7, label="A Val")
    ax.plot(epochs_B, extract(recs_B, tk), color=C_B, lw=2, marker="o", ms=4, label="B Train")
    ax.plot(epochs_B, extract(recs_B, vk), color=C_B, lw=1.5, marker="s", ms=4, ls="--",
            alpha=0.7, label="B Val")
    style_ax(ax, name, ylabel="Loss")

legend_ax = axes[1][2]
legend_ax.axis("off")
legend_ax.legend(
    handles=[
        _mp.Patch(color=C_A, label="Config A — lr=1e-4"),
        _mp.Patch(color=C_B, label="Config B — lr=5e-5"),
        plt.Line2D([0],[0], color="gray", lw=2, label="Train"),
        plt.Line2D([0],[0], color="gray", lw=1.5, ls="--", label="Validation"),
    ],
    loc="center", fontsize=10, title="Legend", title_fontsize=11,
    frameon=True, edgecolor="#CBD5E1",
)

plt.tight_layout()
fig.savefig(RFDETR_DIR / "02_component_losses.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RFDETR_DIR}/02_component_losses.png")

In [ ]:
# ── Chart 3: Detection mAP ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Object Detection Performance — Bounding Box mAP",
             fontsize=15, fontweight="bold")

for ax, (m50_reg, m50_ema, m5095_reg, m5095_ema, ep, c, ce, lbl) in zip(axes, [
    (det_map50_A, det_map50_A_ema, det_map5095_A, det_map5095_A_ema, epochs_A, C_A, C_EMA_A, "A"),
    (det_map50_B, det_map50_B_ema, det_map5095_B, det_map5095_B_ema, epochs_B, C_B, C_EMA_B, "B"),
]):
    ax.plot(ep, m50_reg, color=c,  lw=2,   marker="o", ms=5, ls="--", alpha=0.6, label=f"{lbl} Regular")
    ax.plot(ep, m50_ema, color=ce, lw=2.5, marker="D", ms=6,                     label=f"{lbl} EMA")
    ax.fill_between(ep, m50_reg, m50_ema, alpha=0.08, color=c)
    add_best_marker(ax, ep, m50_ema, ce)
    ax.set_ylim(90, 100)
    style_ax(ax, f"Config {lbl} — Detection mAP@50 (%)", ylabel="mAP@50 (%)")

plt.tight_layout()
fig.savefig(RFDETR_DIR / "03_detection_map.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RFDETR_DIR}/03_detection_map.png")

# ── Chart 4: Segmentation mAP ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Instance Segmentation Performance — Mask mAP",
             fontsize=15, fontweight="bold")

for ax, (m50_reg, m50_ema, ep, c, ce, lbl) in zip(axes, [
    (seg_map50_A, seg_map50_A_ema, epochs_A, C_A, C_EMA_A, "A"),
    (seg_map50_B, seg_map50_B_ema, epochs_B, C_B, C_EMA_B, "B"),
]):
    ax.plot(ep, m50_reg, color=c,  lw=2,   marker="o", ms=5, ls="--", alpha=0.6, label=f"{lbl} Regular")
    ax.plot(ep, m50_ema, color=ce, lw=2.5, marker="D", ms=6,                     label=f"{lbl} EMA")
    ax.fill_between(ep, m50_reg, m50_ema, alpha=0.08, color=c)
    add_best_marker(ax, ep, m50_ema, ce)
    ax.set_ylim(90, 100)
    style_ax(ax, f"Config {lbl} — Segmentation mAP@50 (%)", ylabel="Mask mAP@50 (%)")

plt.tight_layout()
fig.savefig(RFDETR_DIR / "04_segmentation_map.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RFDETR_DIR}/04_segmentation_map.png")

In [ ]:
import matplotlib.patches as _mp2

# ── Chart 5: Precision / Recall / F1 ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Precision, Recall & F1 — Detection vs Segmentation (EMA Model)",
             fontsize=14, fontweight="bold")

metric_sets = [
    ("Detection Precision (%)",     prec_A,    prec_B),
    ("Detection Recall (%)",        rec_A,     rec_B),
    ("Detection F1 Score (%)",      f1_A,      f1_B),
    ("Segmentation Precision (%)",  seg_prec_A, seg_prec_B),
    ("Segmentation Recall (%)",     seg_rec_A,  seg_rec_B),
    ("Segmentation F1 Score (%)",   seg_f1_A,   seg_f1_B),
]
for idx, (title, vals_A, vals_B) in enumerate(metric_sets):
    row, col = divmod(idx, 3)
    ax = axes[row][col]
    if recs_A:
        ax.plot(epochs_A, vals_A, color=C_EMA_A, lw=2.5, marker="D", ms=5, label="Config A (lr=1e-4)")
        ax.fill_between(epochs_A, vals_A, alpha=0.1, color=C_A)
        add_best_marker(ax, epochs_A, vals_A, C_EMA_A)
    ax.plot(epochs_B, vals_B, color=C_EMA_B, lw=2.5, marker="D", ms=5, label="Config B (lr=5e-5)")
    ax.fill_between(epochs_B, vals_B, alpha=0.1, color=C_B)
    add_best_marker(ax, epochs_B, vals_B, C_EMA_B)
    ax.set_ylim(85, 100)
    style_ax(ax, title, ylabel="%")

plt.tight_layout()
fig.savefig(RFDETR_DIR / "05_precision_recall_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RFDETR_DIR}/05_precision_recall_f1.png")

# ── Chart 6: Config A vs B head-to-head bar ─────────────────────────────────
if recs_A and recs_B:
    metric_labels = ["Det mAP@50","Det mAP@50:95","Seg mAP@50","Seg mAP@50:95",
                     "Precision","Recall","F1 Score"]
    last_A = [det_map50_A_ema[-1], det_map5095_A_ema[-1], seg_map50_A_ema[-1],
              seg_map5095_A_ema[-1], prec_A[-1], rec_A[-1], f1_A[-1]]
    last_B = [det_map50_B_ema[-1], det_map5095_B_ema[-1], seg_map50_B_ema[-1],
              seg_map5095_B_ema[-1], prec_B[-1], rec_B[-1], f1_B[-1]]
    x = np.arange(len(metric_labels))
    w = 0.35

    fig, ax = plt.subplots(figsize=(14, 6))
    bars_a = ax.bar(x - w/2, last_A, w, color=C_A, label=f"Config A  lr=1e-4  (Ep {epochs_A[-1]})",
                    edgecolor="white", linewidth=0.8, zorder=3)
    bars_b = ax.bar(x + w/2, last_B, w, color=C_B, label=f"Config B  lr=5e-5  (Ep {epochs_B[-1]})",
                    edgecolor="white", linewidth=0.8, zorder=3)
    for bar in list(bars_a) + list(bars_b):
        h = bar.get_height()
        if h:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.2, f"{h:.1f}",
                    ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(metric_labels, rotation=15, ha="right")
    ax.set_ylabel("Score (%)"); ax.set_ylim(70, 103)
    ax.set_title("Config A vs Config B — Latest Epoch EMA Metrics", fontsize=13, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0); ax.legend(fontsize=10)
    plt.tight_layout()
    fig.savefig(RFDETR_DIR / "06_config_comparison_bar.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {RFDETR_DIR}/06_config_comparison_bar.png")
else:
    print("Skipping Chart 6 (need both Config A and B logs)")

# ── Chart 7: EMA vs Regular Model Gain ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("EMA Model vs Regular Model — mAP Improvement per Epoch",
             fontsize=14, fontweight="bold")

for ax, (m50_reg, m50_ema, ep, c, lbl) in zip(axes, [
    (det_map50_A, det_map50_A_ema, epochs_A, C_A, "Config A"),
    (det_map50_B, det_map50_B_ema, epochs_B, C_B, "Config B"),
]):
    if not ep:
        ax.set_title(f"{lbl} — no data"); ax.axis("off"); continue
    gain = [e - r for e, r in zip(m50_ema, m50_reg) if e and r]
    ep_v  = [e for e, r in zip(ep, m50_reg) if r]
    bars  = ax.bar(ep_v, gain, width=0.6, color=c, alpha=0.8, edgecolor="white")
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"+{h:.2f}",
                ha="center", va="bottom", fontsize=7.5, color=TEXT_COLOR)
    ax.axhline(0, color="#94A3B8", lw=1)
    ax.set_xlabel("Epoch"); ax.set_ylabel("EMA Gain (pp)")
    ax.set_title(f"{lbl} — EMA Benefit over Regular Model", fontweight="bold")
    ax.set_xticks(ep_v); ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
fig.savefig(RFDETR_DIR / "07_ema_gain.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RFDETR_DIR}/07_ema_gain.png")

In [ ]:
# ── Chart 8: Epoch Training Time ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
if recs_A and train_min_A:
    ax.plot(epochs_A, train_min_A, color=C_A, lw=2.5, marker="o", ms=7, label="Config A  lr=1e-4")
ax.plot(epochs_B, train_min_B, color=C_B, lw=2.5, marker="o", ms=7, label="Config B  lr=5e-5")
ax.set_xlabel("Epoch"); ax.set_ylabel("Training Time (minutes)")
ax.set_title("Training Time per Epoch", fontweight="bold")
ax.grid(axis="y", linestyle="--", alpha=0.5); ax.legend()
plt.tight_layout()
fig.savefig(RFDETR_DIR / "08_epoch_training_time.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RFDETR_DIR}/08_epoch_training_time.png")

# ── Chart 9: Comprehensive Dashboard ────────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor("white")
gs = GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

fig.text(0.5, 0.98, "RF-DETR-Seg Hyperparameter Comparison — Training Dashboard",
         ha="center", va="top", fontsize=17, fontweight="bold", color=TEXT_COLOR)
fig.text(0.5, 0.955, "Texas State University · Cattle Vision Framework · Master's Thesis",
         ha="center", va="top", fontsize=10, color="#64748B")

ax1 = fig.add_subplot(gs[0, :2])
if recs_A:
    ax1.plot(epochs_A, train_loss_A, color=C_A, lw=2, marker="o", ms=4, label="A Train")
    ax1.plot(epochs_A, test_loss_A,  color=C_A, lw=1.5, ls="--", marker="s", ms=4, alpha=0.7, label="A Val")
ax1.plot(epochs_B, train_loss_B, color=C_B, lw=2, marker="o", ms=4, label="B Train")
ax1.plot(epochs_B, test_loss_B,  color=C_B, lw=1.5, ls="--", marker="s", ms=4, alpha=0.7, label="B Val")
style_ax(ax1, "Total Loss", ylabel="Loss")

ax2 = fig.add_subplot(gs[0, 2])
if recs_A:
    ax2.plot(epochs_A, det_map50_A_ema, color=C_EMA_A, lw=2, marker="D", ms=5, label="A EMA")
ax2.plot(epochs_B, det_map50_B_ema, color=C_EMA_B, lw=2, marker="D", ms=5, label="B EMA")
ax2.set_ylim(90, 100); style_ax(ax2, "Det mAP@50 (%)", ylabel="%")

ax3 = fig.add_subplot(gs[0, 3])
if recs_A:
    ax3.plot(epochs_A, seg_map5095_A_ema, color=C_EMA_A, lw=2, marker="D", ms=5, label="A EMA")
ax3.plot(epochs_B, seg_map5095_B_ema, color=C_EMA_B, lw=2, marker="D", ms=5, label="B EMA")
style_ax(ax3, "Seg mAP@50:95 (%)", ylabel="%")

ax4 = fig.add_subplot(gs[1, 0])
if recs_A:
    ax4.plot(epochs_A, extract(recs_A, "train_loss_ce"), color=C_A, lw=2, marker="o", ms=4, label="A")
ax4.plot(epochs_B, extract(recs_B, "train_loss_ce"), color=C_B, lw=2, marker="o", ms=4, label="B")
style_ax(ax4, "Train CE Loss", ylabel="Loss")

ax5 = fig.add_subplot(gs[1, 1])
if recs_A:
    ax5.plot(epochs_A, extract(recs_A, "train_loss_giou"), color=C_A, lw=2, marker="o", ms=4, label="A")
ax5.plot(epochs_B, extract(recs_B, "train_loss_giou"), color=C_B, lw=2, marker="o", ms=4, label="B")
style_ax(ax5, "Train GIoU Loss", ylabel="Loss")

ax6 = fig.add_subplot(gs[1, 2])
if recs_A:
    ax6.plot(epochs_A, prec_A, color=C_EMA_A, lw=2, marker="D", ms=5, label="A Precision")
    ax6.plot(epochs_A, rec_A,  color=C_EMA_A, lw=2, marker="s", ms=5, ls="--", label="A Recall")
ax6.plot(epochs_B, prec_B, color=C_EMA_B, lw=2, marker="D", ms=5, label="B Precision")
ax6.plot(epochs_B, rec_B,  color=C_EMA_B, lw=2, marker="s", ms=5, ls="--", label="B Recall")
ax6.set_ylim(85, 100); style_ax(ax6, "Detection Precision & Recall (%)", ylabel="%")

ax7 = fig.add_subplot(gs[1, 3])
if recs_A:
    ax7.plot(epochs_A, f1_A,     color=C_EMA_A, lw=2.5, marker="D", ms=5, label="A Det F1")
    ax7.plot(epochs_A, seg_f1_A, color=C_A,     lw=2,   marker="o", ms=5, ls="--", label="A Seg F1")
ax7.plot(epochs_B, f1_B,     color=C_EMA_B, lw=2.5, marker="D", ms=5, label="B Det F1")
ax7.plot(epochs_B, seg_f1_B, color=C_B,     lw=2,   marker="o", ms=5, ls="--", label="B Seg F1")
ax7.set_ylim(85, 100); style_ax(ax7, "F1 Score — Det vs Seg (%)", ylabel="%")

# Summary text panel
ax8 = fig.add_subplot(gs[2, :])
ax8.axis("off")
summary_lines = [
    f"Config B (lr=5e-5): {len(recs_B)} epochs loaded from logs/train_rfdetr_seg_hipe1.log",
    f"Config A (lr=1e-4): {len(recs_A)} epochs loaded" if recs_A else "Config A: not loaded (LOG_A = None)",
    f"Outputs saved to: {RFDETR_DIR}",
]
ax8.text(0.5, 0.5, "\n".join(summary_lines), transform=ax8.transAxes,
         ha="center", va="center", fontsize=11, color=TEXT_COLOR,
         bbox=dict(boxstyle="round,pad=0.5", facecolor=BG, edgecolor="#CBD5E1"))

plt.savefig(RFDETR_DIR / "09_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RFDETR_DIR}/09_dashboard.png")

print(f"\n✅ All RF-DETR-Seg charts saved to {RFDETR_DIR}")
charts = sorted(RFDETR_DIR.glob("*.png"))
for p in charts:
    print(f"  {p.name:<45s} {p.stat().st_size/1024:6.1f} KB")

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import numpy as np

REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
LOGS_DIR = os.path.join(REPO_ROOT, "results", "behavior", "training_logs")
F1_CSV = os.path.join(REPO_ROOT, "results", "behavior", "f1_per_class.csv")
OUT_DIR = os.path.join(REPO_ROOT, "results", "figures", "graphs")
os.makedirs(OUT_DIR, exist_ok=True)

WRITTEN_FILES = []


def apply_style(ax, title, xlabel, ylabel):
    """Shared style helper — seaborn base, consistent font sizes."""
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.tick_params(labelsize=10)
    ax.legend(fontsize=10)
    sns.despine(ax=ax)


def save_fig(fig, filename):
    path = os.path.join(OUT_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    WRITTEN_FILES.append(path)
    plt.close(fig)


sns.set_theme(style="whitegrid")
matplotlib.rcParams["figure.dpi"] = 100
print("Setup complete. Output dir:", OUT_DIR)

Setup complete. Output dir: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/graphs


## Section 1 — Training Curves

For each of the 5 behavior configs, plot `train_loss`, `val_loss`, and `val_macro_f1` vs epoch.  
v1 and v2 variants are overlaid on the same axes.


In [2]:
CONFIGS = [
    ("cbvd5", "CBVD-5 In-Domain"),
    ("cvb", "CVB In-Domain"),
    ("cbvd5_to_cvb", "CBVD-5 → CVB Cross-Domain"),
    ("cvb_to_cbvd5", "CVB → CBVD-5 Cross-Domain"),
    ("combined", "Combined"),
]

V1_STYLE = dict(linestyle="-", linewidth=1.8)
V2_STYLE = dict(linestyle="--", linewidth=1.8)

COLORS = {"train_loss": "#e07b39", "val_loss": "#4878d0", "val_macro_f1": "#6acc65"}

for key, label in CONFIGS:
    v1_path = os.path.join(LOGS_DIR, f"videomae_{key}_v1.csv")
    v2_path = os.path.join(LOGS_DIR, f"videomae_{key}_v2.csv")

    df1 = pd.read_csv(v1_path)
    df2 = pd.read_csv(v2_path)
    print(f"{label} — v1 shape: {df1.shape}, v2 shape: {df2.shape}")

    fig, axes = plt.subplots(1, 2, figsize=(8, 5))
    fig.suptitle(f"Training Curves — {label}", fontsize=14, y=1.01)

    # Loss subplot
    ax = axes[0]
    ax.plot(
        df1["epoch"],
        df1["train_loss"],
        color=COLORS["train_loss"],
        label="train_loss v1",
        **V1_STYLE,
    )
    ax.plot(
        df1["epoch"],
        df1["val_loss"],
        color=COLORS["val_loss"],
        label="val_loss v1",
        **V1_STYLE,
    )
    ax.plot(
        df2["epoch"],
        df2["train_loss"],
        color=COLORS["train_loss"],
        label="train_loss v2",
        **V2_STYLE,
    )
    ax.plot(
        df2["epoch"],
        df2["val_loss"],
        color=COLORS["val_loss"],
        label="val_loss v2",
        **V2_STYLE,
    )
    apply_style(ax, "Loss", "Epoch", "Loss")

    # Macro-F1 subplot
    ax = axes[1]
    ax.plot(
        df1["epoch"],
        df1["val_macro_f1"],
        color=COLORS["val_macro_f1"],
        label="val_macro_f1 v1",
        **V1_STYLE,
    )
    ax.plot(
        df2["epoch"],
        df2["val_macro_f1"],
        color=COLORS["val_macro_f1"],
        label="val_macro_f1 v2",
        **V2_STYLE,
    )
    ax.set_ylim(0, 1)
    apply_style(ax, "Val Macro-F1", "Epoch", "Macro-F1")

    fig.tight_layout()
    save_fig(fig, f"training_curves_{key}.png")
    print(f"  → saved training_curves_{key}.png")

CBVD-5 In-Domain — v1 shape: (12, 12), v2 shape: (9, 12)
  → saved training_curves_cbvd5.png
CVB In-Domain — v1 shape: (12, 12), v2 shape: (10, 12)
  → saved training_curves_cvb.png
CBVD-5 → CVB Cross-Domain — v1 shape: (12, 12), v2 shape: (19, 12)
  → saved training_curves_cbvd5_to_cvb.png
CVB → CBVD-5 Cross-Domain — v1 shape: (15, 12), v2 shape: (12, 12)
  → saved training_curves_cvb_to_cbvd5.png
Combined — v1 shape: (24, 12), v2 shape: (16, 12)
  → saved training_curves_combined.png


## Section 2 — Per-Class F1 Comparison

Grouped bar chart: 7 behavior classes on x-axis, 5 configs × v1/v2 as bar groups.


In [3]:
df_f1 = pd.read_csv(F1_CSV)
print(f"f1_per_class.csv shape: {df_f1.shape}")

CLASS_COLS = [
    "f1_standing",
    "f1_lying",
    "f1_foraging",
    "f1_drinking",
    "f1_ruminating",
    "f1_grooming",
    "f1_other",
]
CLASS_LABELS = [
    "Standing",
    "Lying",
    "Foraging",
    "Drinking",
    "Ruminating",
    "Grooming",
    "Other",
]

CONFIG_ORDER = [
    ("videomae_cbvd5_v1", "CBVD-5 v1"),
    ("videomae_cbvd5_v2", "CBVD-5 v2"),
    ("videomae_cvb_v1", "CVB v1"),
    ("videomae_cvb_v2", "CVB v2"),
    ("videomae_cbvd5_to_cvb_v1", "CBVD5→CVB v1"),
    ("videomae_cbvd5_to_cvb_v2", "CBVD5→CVB v2"),
    ("videomae_cvb_to_cbvd5_v1", "CVB→CBVD5 v1"),
    ("videomae_cvb_to_cbvd5_v2", "CVB→CBVD5 v2"),
    ("videomae_combined_v1", "Combined v1"),
    ("videomae_combined_v2", "Combined v2"),
]

df_f1 = df_f1.set_index("experiment")
rows = []
for exp_key, exp_label in CONFIG_ORDER:
    if exp_key in df_f1.index:
        row = df_f1.loc[exp_key, CLASS_COLS].values.astype(float)
        rows.append({"label": exp_label, **dict(zip(CLASS_LABELS, row))})

plot_df = pd.DataFrame(rows).set_index("label")

n_groups = len(CLASS_LABELS)
n_bars = len(plot_df)
x = np.arange(n_groups)
width = 0.08
offsets = np.linspace(-(n_bars - 1) / 2, (n_bars - 1) / 2, n_bars) * width

palette = sns.color_palette("tab10", n_bars)

fig, ax = plt.subplots(figsize=(14, 6))
for i, (label, row) in enumerate(plot_df.iterrows()):
    ax.bar(
        x + offsets[i],
        row.values,
        width,
        label=label,
        color=palette[i],
        alpha=0.85,
        edgecolor="white",
        linewidth=0.4,
    )

ax.set_xticks(x)
ax.set_xticklabels(CLASS_LABELS, fontsize=10)
ax.set_ylim(0, 1.05)
apply_style(ax, "Per-Class F1 — All Configs (v1 vs v2)", "Behavior Class", "F1 Score")
ax.legend(fontsize=8, ncol=2, loc="upper right")

fig.tight_layout()
save_fig(fig, "f1_per_class_comparison.png")
print("→ saved f1_per_class_comparison.png")

f1_per_class.csv shape: (10, 24)
→ saved f1_per_class_comparison.png


## Section 3 — Cross-Dataset mAP50 Comparison

Bar chart of mAP50 across all six datasets, with mAP75 and AR@100 as secondary bars.
Source: `results/detection/*.json` merged with `results/generalization/ood_summary.csv`.


In [4]:
import json
import glob

DETECTION_DIR = os.path.join(REPO_ROOT, "results", "detection")
OOD_CSV = os.path.join(REPO_ROOT, "results", "generalization", "ood_summary.csv")

# Load in-domain results from individual JSONs
in_domain_files = {
    "cbvd5": os.path.join(DETECTION_DIR, "cbvd5_test_ap.json"),
    "cvb": os.path.join(DETECTION_DIR, "cvb_test_ap.json"),
}
rows = []
for ds, path in in_domain_files.items():
    d = json.load(open(path))
    rows.append(
        {
            "dataset": ds,
            "mAP50": d["mAP50"],
            "mAP75": d.get("mAP75", float("nan")),
            "AR100": d.get("AR100", float("nan")),
            "split": "in-domain",
        }
    )

# Load OOD results from CSV
df_ood = pd.read_csv(OOD_CSV)
print(f"ood_summary.csv shape: {df_ood.shape}")
for _, row in df_ood.iterrows():
    rows.append(
        {
            "dataset": row["dataset"],
            "mAP50": row["mAP50"],
            "mAP75": row.get("mAP50_95", float("nan")),  # mAP50_95 is closest available
            "AR100": row["AR100"],
            "split": "OOD",
        }
    )

df_det = pd.DataFrame(rows)
print(f"combined detection df shape: {df_det.shape}")
print(df_det[["dataset", "split", "mAP50", "mAP75", "AR100"]].to_string(index=False))

DATASET_LABELS = {
    "cbvd5": "CBVD-5\n(in-domain)",
    "cvb": "CVB\n(in-domain)",
    "opencows2020": "OpenCows\n2020",
    "cows2021": "Cows2021",
    "cattleeyeview": "CattleEye\nView",
    "freeman": "Freeman\nCenter",
}
DATASET_ORDER = ["cbvd5", "cvb", "opencows2020", "cows2021", "cattleeyeview", "freeman"]

df_det = df_det.set_index("dataset").loc[DATASET_ORDER].reset_index()
labels = [DATASET_LABELS[d] for d in DATASET_ORDER]

x = np.arange(len(DATASET_ORDER))
w = 0.26
colors = sns.color_palette("Set2", 3)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, df_det["mAP50"], w, label="mAP@50", color=colors[0], edgecolor="white")
ax.bar(x, df_det["mAP75"], w, label="mAP@75", color=colors[1], edgecolor="white")
ax.bar(x + w, df_det["AR100"], w, label="AR@100", color=colors[2], edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1.0)
ax.axvline(
    1.5, color="gray", linestyle=":", linewidth=1.2, label="in-domain / OOD boundary"
)
apply_style(ax, "Cross-Dataset Detection Performance", "Dataset", "Score")

fig.tight_layout()
save_fig(fig, "cross_dataset_detection.png")
print("→ saved cross_dataset_detection.png")

ood_summary.csv shape: (4, 10)
combined detection df shape: (6, 5)
      dataset     split  mAP50  mAP75  AR100
        cbvd5 in-domain 0.4591 0.0552 0.2428
          cvb in-domain 0.0567 0.0319 0.0431
 opencows2020       OOD 0.3326 0.0705 0.2101
     cows2021       OOD 0.2729 0.0624 0.2221
cattleeyeview       OOD 0.4700 0.3209 0.4961
      freeman       OOD 0.7298 0.4430 0.4991
→ saved cross_dataset_detection.png


## Section 4 — Perturbation Robustness Heatmap

Two side-by-side heatmaps (low severity / high severity).  
Axes: 5 perturbation types × 6 datasets. Cells annotated with Δ mAP50.  
Source: `results/generalization/perturbation_delta.csv`.


In [5]:
PERT_CSV = os.path.join(
    REPO_ROOT, "results", "generalization", "perturbation_delta.csv"
)
df_pert = pd.read_csv(PERT_CSV)
print(f"perturbation_delta.csv shape: {df_pert.shape}")
print(
    df_pert[["dataset", "perturbation_type", "severity", "delta_mAP50"]]
    .head(10)
    .to_string(index=False)
)

PERT_ORDER = ["brightness", "gaussian_noise", "motion_blur", "fog", "rain"]
PERT_LABELS = {
    "brightness": "Brightness",
    "gaussian_noise": "Noise",
    "motion_blur": "Blur",
    "fog": "Fog",
    "rain": "Rain",
}
DS_ORDER = ["cbvd5", "cvb", "opencows2020", "cows2021", "cattleeyeview", "freeman"]
DS_LABELS = ["CBVD-5", "CVB", "OpenCows", "Cows2021", "CattleEye", "Freeman"]


def make_pivot(severity):
    sub = df_pert[df_pert["severity"] == severity].copy()
    sub = sub[sub["perturbation_type"].isin(PERT_ORDER) & sub["dataset"].isin(DS_ORDER)]
    piv = sub.pivot(index="perturbation_type", columns="dataset", values="delta_mAP50")
    piv = piv.loc[PERT_ORDER, DS_ORDER]
    piv.index = [PERT_LABELS[p] for p in PERT_ORDER]
    piv.columns = DS_LABELS
    return piv


piv_low = make_pivot("low")
piv_high = make_pivot("high")

vmax = max(piv_low.abs().max().max(), piv_high.abs().max().max())

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, piv, title in zip(axes, [piv_low, piv_high], ["Low Severity", "High Severity"]):
    sns.heatmap(
        piv,
        ax=ax,
        cmap="RdYlGn",
        center=0,
        vmin=-vmax,
        vmax=0,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 9},
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Δ mAP50"},
    )
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Dataset", fontsize=12)
    ax.set_ylabel("Perturbation" if ax is axes[0] else "", fontsize=12)
    ax.tick_params(labelsize=10)

fig.suptitle("Perturbation Robustness — Δ mAP50 vs Clean", fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, "perturbation_robustness_heatmap.png")
print("→ saved perturbation_robustness_heatmap.png")

perturbation_delta.csv shape: (60, 11)
dataset perturbation_type severity  delta_mAP50
  cbvd5        brightness      low      -0.3723
  cbvd5        brightness     high      -0.4473
  cbvd5    gaussian_noise      low      -0.0254
  cbvd5    gaussian_noise     high      -0.1083
  cbvd5       motion_blur      low       0.0026
  cbvd5       motion_blur     high      -0.0244
  cbvd5               fog      low      -0.1735
  cbvd5               fog     high      -0.2419
  cbvd5              rain      low      -0.0219
  cbvd5              rain     high      -0.0751
→ saved perturbation_robustness_heatmap.png


## Section 5 — Per-Video Tracking Distribution

Side-by-side box plots of IDF1 and MOTA across all 447 CVB evaluation videos,
with individual video points overlaid as a strip plot.  
Source: `results/tracking/tracking_per_video_all.csv`.


In [6]:
TRACKING_DIR = os.path.join(REPO_ROOT, "results", "tracking")
PER_VIDEO_CSV = os.path.join(TRACKING_DIR, "tracking_per_video_all.csv")

df_pv = pd.read_csv(PER_VIDEO_CSV)
print(f"tracking_per_video_all.csv shape: {df_pv.shape}")
print(df_pv[["idf1", "mota"]].describe().round(2))

# Melt to long form for easy plotting
df_long = df_pv[["idf1", "mota"]].melt(var_name="Metric", value_name="Value")
df_long["Metric"] = df_long["Metric"].map({"idf1": "IDF1", "mota": "MOTA"})

rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(8, 5))
palette = {"IDF1": "#4878d0", "MOTA": "#e07b39"}

sns.boxplot(
    data=df_long,
    x="Metric",
    y="Value",
    ax=ax,
    palette=palette,
    width=0.4,
    fliersize=0,
    boxprops=dict(alpha=0.7),
)
sns.stripplot(
    data=df_long,
    x="Metric",
    y="Value",
    ax=ax,
    palette=palette,
    size=2.5,
    alpha=0.35,
    jitter=True,
)

apply_style(ax, "Per-Video Tracking Distribution (CVB, n=447)", "Metric", "Score (%)")
ax.set_ylim(df_long["Value"].min() - 5, 105)

# Annotate overall summary values
for metric, val, xpos in [
    ("IDF1", df_pv["idf1"].mean(), 0),
    ("MOTA", df_pv["mota"].mean(), 1),
]:
    ax.text(
        xpos,
        df_long[df_long["Metric"] == metric]["Value"].max() + 2,
        f"mean={val:.1f}",
        ha="center",
        fontsize=9,
        color="#333333",
    )

fig.tight_layout()
save_fig(fig, "tracking_per_video_distribution.png")
print("→ saved tracking_per_video_distribution.png")

tracking_per_video_all.csv shape: (447, 13)
         idf1     mota
count  447.00   447.00
mean    67.31    14.14
std     21.13    98.33
min      0.00 -1000.00
25%     55.12     0.00
50%     70.22    37.14
75%     82.35    63.57
max    100.00   100.00
→ saved tracking_per_video_distribution.png


/tmp/ipykernel_472824/1316250346.py:17: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_472824/1316250346.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(
/tmp/ipykernel_472824/1057673989.py:23: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=10)


## Section 6 — min_hits Ablation

Grouped bar chart: IDF1, MOTA, and total ID switches for min_hits ∈ {1, 2, 3, 5}.  
Source: `results/tracking/minhits_ablation.csv`.


In [7]:
MINHITS_CSV = os.path.join(TRACKING_DIR, "minhits_ablation.csv")

df_mh = pd.read_csv(MINHITS_CSV)
print(f"minhits_ablation.csv shape: {df_mh.shape}")
print(df_mh.to_string(index=False))

x = np.arange(len(df_mh))
w = 0.22
colors = sns.color_palette("Set2", 3)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

b1 = ax1.bar(x - w, df_mh["idf1"], w, label="IDF1", color=colors[0], edgecolor="white")
b2 = ax1.bar(x, df_mh["mota"], w, label="MOTA", color=colors[1], edgecolor="white")
b3 = ax2.bar(
    x + w,
    df_mh["total_id_switches"],
    w,
    label="ID Switches",
    color=colors[2],
    edgecolor="white",
    alpha=0.8,
)

ax1.set_xticks(x)
ax1.set_xticklabels([f"min_hits={v}" for v in df_mh["min_hits"]], fontsize=10)
ax1.set_ylim(0, 100)
ax1.set_ylabel("Score (%)", fontsize=12)
ax1.tick_params(labelsize=10)
ax2.set_ylabel("Total ID Switches", fontsize=12)
ax2.tick_params(labelsize=10)

# Annotate invariance
ax1.set_title(
    "min_hits Ablation — IDF1, MOTA, and ID Switches (all invariant)", fontsize=14
)
ax1.set_xlabel("min_hits Setting", fontsize=12)
sns.despine(ax=ax1, right=False)

lines = [b1, b2, b3]
ax1.legend(lines, ["IDF1", "MOTA", "ID Switches"], fontsize=10, loc="lower right")

# Annotate bars with values
for bar in b1:
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{bar.get_height():.1f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )
for bar in b2:
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{bar.get_height():.1f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )
for bar in b3:
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{int(bar.get_height())}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

fig.tight_layout()
save_fig(fig, "tracking_minhits_ablation.png")
print("→ saved tracking_minhits_ablation.png")

minhits_ablation.csv shape: (4, 6)
 min_hits  idf1  mota  motp  total_id_switches  n_videos
        1 67.31 36.60 77.42                141       447
        2 67.31 36.61 77.42                141       447
        3 67.31 36.61 77.42                141       447
        5 67.31 36.61 77.42                141       447
→ saved tracking_minhits_ablation.png


## Section 7 — Activity Budget

Stacked bar chart of mean percentage time per behavior class, one bar per dataset.  
Source: `results/behavior_analytics/activity_budget.csv`.


In [8]:
ANALYTICS_DIR = os.path.join(REPO_ROOT, "results", "analytics")
BUDGET_CSV = os.path.join(ANALYTICS_DIR, "activity_budget.csv")

df_budget = pd.read_csv(BUDGET_CSV)
print(f"activity_budget.csv shape: {df_budget.shape}")

BEHAVIOR_ORDER = [
    "Standing",
    "Lying",
    "Foraging",
    "Drinking",
    "Ruminating",
    "Grooming",
    "Other",
]
BEHAVIOR_COLORS = sns.color_palette("tab10", len(BEHAVIOR_ORDER))

# Mean pct_time per dataset × behavior
pivot = (
    df_budget.groupby(["dataset", "behavior"])["pct_time"]
    .mean()
    .unstack("behavior")
    .reindex(columns=BEHAVIOR_ORDER, fill_value=0)
)
print(pivot.round(2))

fig, ax = plt.subplots(figsize=(8, 5))
bottom = np.zeros(len(pivot))
for j, beh in enumerate(BEHAVIOR_ORDER):
    vals = pivot[beh].values
    bars = ax.bar(
        pivot.index,
        vals,
        bottom=bottom,
        label=beh,
        color=BEHAVIOR_COLORS[j],
        edgecolor="white",
        linewidth=0.4,
    )
    # Label segments > 5%
    for bar, v, b in zip(bars, vals, bottom):
        if v > 5:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                b + v / 2,
                f"{v:.0f}%",
                ha="center",
                va="center",
                fontsize=8,
                color="white",
                fontweight="bold",
            )
    bottom += vals

ax.set_ylim(0, 110)
apply_style(ax, "Mean Activity Budget per Dataset", "Dataset", "Mean % Time")
ax.legend(loc="upper right", fontsize=9, ncol=2)

fig.tight_layout()
save_fig(fig, "activity_budget.png")
print("→ saved activity_budget.png")

activity_budget.csv shape: (4879, 7)
behavior  Standing  Lying  Foraging  Drinking  Ruminating  Grooming  Other
dataset                                                                   
cbvd5        52.60   4.55     28.57       3.9       10.39       0.0   0.00
cvb          11.57  16.08     49.19       3.8       13.93       1.1   4.33
→ saved activity_budget.png


## Section 8 — Behavior Transition Heatmap

Matrix of transition probabilities (from_behavior × to_behavior), annotated with probability values.  
Source: `results/behavior_analytics/transition_matrix.csv`.


In [9]:
TRANSITION_CSV = os.path.join(ANALYTICS_DIR, "transition_matrix.csv")

df_trans = pd.read_csv(TRANSITION_CSV)
print(f"transition_matrix.csv shape: {df_trans.shape}")
print("datasets present:", df_trans["dataset"].unique())

BEH_ORDER = [
    "Standing",
    "Lying",
    "Foraging",
    "Drinking",
    "Ruminating",
    "Grooming",
    "Other",
]

# Use CVB (only dataset available)
sub = df_trans[df_trans["dataset"] == "cvb"]
piv_t = sub.pivot(
    index="from_behavior", columns="to_behavior", values="probability"
).reindex(index=BEH_ORDER, columns=BEH_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    piv_t,
    ax=ax,
    cmap="Blues",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 9},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Transition Probability"},
)
ax.set_title("Behavior Transition Probabilities (CVB)", fontsize=14)
ax.set_xlabel("To Behavior", fontsize=12)
ax.set_ylabel("From Behavior", fontsize=12)
ax.tick_params(labelsize=10)

fig.tight_layout()
save_fig(fig, "behavior_transitions.png")
print("→ saved behavior_transitions.png")

transition_matrix.csv shape: (37, 7)
datasets present: <StringArray>
['cvb']
Length: 1, dtype: str
→ saved behavior_transitions.png


## Section 9 — Behavioral Deviation

Box plot of deviation per behavior class with outlier points flagged in a distinct color.  
Source: `results/behavior_analytics/behavior_deviation.csv`.


In [10]:
DEVIATION_CSV = os.path.join(ANALYTICS_DIR, "behavior_deviation.csv")

df_dev = pd.read_csv(DEVIATION_CSV)
print(f"behavior_deviation.csv shape: {df_dev.shape}")
print("is_outlier distribution:", df_dev["is_outlier"].value_counts().to_dict())

BEH_ORDER = [
    "Standing",
    "Lying",
    "Foraging",
    "Drinking",
    "Ruminating",
    "Grooming",
    "Other",
]
df_dev["behavior"] = pd.Categorical(
    df_dev["behavior"], categories=BEH_ORDER, ordered=True
)
df_dev = df_dev.sort_values("behavior")

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=df_dev,
    x="behavior",
    y="deviation",
    ax=ax,
    color="#4878d0",
    width=0.5,
    fliersize=0,
    boxprops=dict(alpha=0.7),
)
# Overlay outliers in red (if any), non-outliers as small grey dots
non_out = df_dev[~df_dev["is_outlier"]]
out = df_dev[df_dev["is_outlier"]]

if not non_out.empty:
    sns.stripplot(
        data=non_out,
        x="behavior",
        y="deviation",
        ax=ax,
        color="#aaaaaa",
        size=2,
        alpha=0.3,
        jitter=True,
        order=BEH_ORDER,
    )
if not out.empty:
    sns.stripplot(
        data=out,
        x="behavior",
        y="deviation",
        ax=ax,
        color="#e07b39",
        size=5,
        alpha=0.8,
        jitter=True,
        order=BEH_ORDER,
        label="Outlier",
    )
    ax.legend(fontsize=10)

apply_style(
    ax, "Behavioral Deviation per Class", "Behavior", "Deviation (pp from baseline)"
)
ax.set_xticklabels(BEH_ORDER, rotation=15, ha="right", fontsize=10)

fig.tight_layout()
save_fig(fig, "behavior_deviation.png")
print("→ saved behavior_deviation.png")

behavior_deviation.csv shape: (4879, 9)
is_outlier distribution: {False: 4879}
→ saved behavior_deviation.png


/tmp/ipykernel_472824/1057673989.py:23: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=10)
/tmp/ipykernel_472824/458788881.py:66: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(BEH_ORDER, rotation=15, ha="right", fontsize=10)


## Section 10 — Pipeline Timing Breakdown

Horizontal stacked bar showing seconds per stage for the Freeman Center pipeline run.  
Source: `results/inference/timing.csv`.


In [11]:
TIMING_CSV = os.path.join(REPO_ROOT, "results", "inference", "timing.csv")

df_time = pd.read_csv(TIMING_CSV)
print(f"timing.csv shape: {df_time.shape}")
print(df_time.to_string(index=False))

STAGES = [
    "ingest_s",
    "detect_s",
    "track_s",
    "extract_s",
    "classify_s",
    "analyze_s",
    "render_s",
]
LABELS = ["Ingest", "Detect", "Track", "Extract", "Classify", "Analyze", "Render"]
palette = sns.color_palette("tab10", len(STAGES))

# Sum across all videos
totals = df_time[STAGES].sum()

fig, ax = plt.subplots(figsize=(10, 3))
left = 0
for stage, label, color in zip(STAGES, LABELS, palette):
    val = totals[stage]
    ax.barh(0, val, left=left, color=color, edgecolor="white", label=label)
    if val > totals.sum() * 0.02:  # label if > 2% of total
        ax.text(
            left + val / 2,
            0,
            f"{label}\n{val:.0f}s",
            ha="center",
            va="center",
            fontsize=8,
            color="white",
            fontweight="bold",
        )
    left += val

ax.set_yticks([])
ax.set_xlabel("Wall-clock Time (s)", fontsize=12)
ax.set_title("Freeman Center Pipeline Timing Breakdown", fontsize=14)
ax.tick_params(labelsize=10)
ax.legend(loc="lower right", fontsize=9, ncol=4, bbox_to_anchor=(1, 1))
sns.despine(ax=ax, left=True)

fig.tight_layout()
save_fig(fig, "pipeline_timing.png")
print("→ saved pipeline_timing.png")

timing.csv shape: (1, 13)
  video  file_size_mb  frame_count  fps  duration_s  ingest_s  detect_s  track_s  extract_s  classify_s  analyze_s  render_s  total_s
640.mp4          3.44          250 25.0        10.0      0.04     22.29     0.26     289.33       15.41       0.07      7.31   334.71
→ saved pipeline_timing.png


## Section 11 — Segmentation Coverage Histograms

Overlaid histograms of coverage_rate for CBVD-5 and CVB.  
Source: `results/segmentation/{cbvd5,cvb}_segmentation_stats.csv`.


In [12]:
SEG_DIR = os.path.join(REPO_ROOT, "results", "segmentation")

df_seg_cbvd5 = pd.read_csv(os.path.join(SEG_DIR, "cbvd5_segmentation_stats.csv"))
df_seg_cvb = pd.read_csv(os.path.join(SEG_DIR, "cvb_segmentation_stats.csv"))
print(f"cbvd5_segmentation_stats.csv shape: {df_seg_cbvd5.shape}")
print(f"cvb_segmentation_stats.csv shape:   {df_seg_cvb.shape}")
print("CBVD-5 coverage_rate describe:")
print(df_seg_cbvd5["coverage_rate"].describe().round(3))
print("CVB coverage_rate describe:")
print(df_seg_cvb["coverage_rate"].describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)

for ax, df, ds, color in zip(
    axes,
    [df_seg_cbvd5, df_seg_cvb],
    ["CBVD-5", "CVB"],
    ["#4878d0", "#e07b39"],
):
    ax.hist(df["coverage_rate"], bins=30, color=color, alpha=0.75, edgecolor="white")
    ax.axvline(
        df["coverage_rate"].mean(),
        color="black",
        linestyle="--",
        linewidth=1.2,
        label=f'mean={df["coverage_rate"].mean():.2f}',
    )
    apply_style(ax, f"Segmentation Coverage — {ds}", "Coverage Rate", "Count")
    ax.legend(fontsize=9)

fig.tight_layout()
save_fig(fig, "segmentation_coverage.png")
print("→ saved segmentation_coverage.png")

cbvd5_segmentation_stats.csv shape: (684, 11)
cvb_segmentation_stats.csv shape:   (95, 11)
CBVD-5 coverage_rate describe:
count    684.000
mean       0.873
std        0.333
min        0.000
25%        1.000
50%        1.000
75%        1.000
max        1.000
Name: coverage_rate, dtype: float64
CVB coverage_rate describe:
count    95.000
mean      1.368
std       1.440
min       0.000
25%       0.991
50%       1.003
75%       1.016
max      10.000
Name: coverage_rate, dtype: float64
→ saved segmentation_coverage.png


## Written Files


In [13]:
print(f"Files written to {OUT_DIR}:\n")
for path in sorted(WRITTEN_FILES):
    size_kb = os.path.getsize(path) / 1024
    print(f"  {os.path.basename(path):45s}  {size_kb:7.1f} KB")

Files written to /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/graphs:

  activity_budget.png                               49.7 KB
  behavior_deviation.png                            68.8 KB
  behavior_transitions.png                          92.9 KB
  cross_dataset_detection.png                       64.0 KB
  f1_per_class_comparison.png                       65.4 KB
  perturbation_robustness_heatmap.png              108.9 KB
  pipeline_timing.png                               38.4 KB
  segmentation_coverage.png                         48.2 KB
  tracking_minhits_ablation.png                     58.4 KB
  tracking_per_video_distribution.png               68.5 KB
  training_curves_cbvd5.png                         90.8 KB
  training_curves_cbvd5_to_cvb.png                  91.7 KB
  training_curves_combined.png                     121.1 KB
  training_curves_cvb.png                           89.7 KB
  training_curves_cvb_to_cbvd5.png                  93.1 KB
